# Model H — Hindi Pretraining (Colab)

25,379,840-parameter decoder-only Transformer, 467,874,360 training tokens, one
pass = **14,329 steps** at 64 x 512 = 32,768 tokens/step.

**Runtime: GPU (T4).** Set it before running anything.

**If the session drops, just re-run this notebook.** The trainer finds
`checkpoint_latest.pt` on Drive and resumes from it — model, optimizer,
scheduler, GradScaler, RNG. A disconnect costs at most 500 steps (~3 min).

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0))

## 2. Mount Drive and load the code

In [ ]:
import shutil, os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/24KaratAu'
WORK = '/content/phase2'

if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(f'{DRIVE_BASE}/code', WORK)
%cd {WORK}

In [ ]:
!pip install -q pyyaml

## 3. Sanity tests

15 tests: causal masking, parameter count, checkpoint resume, dataset contract.
**Do not proceed if any fail** — a broken causal mask trains normally and
reports a great validation loss while quietly copying from the future.

In [ ]:
!pip install -q pytest && python -m pytest tests/ -q

## 4. Smoke run — 100 steps

Catches a broken config, an OOM, or a dead loss curve in ~1 minute instead of
two hours in. Writes to a throwaway directory so it cannot touch the real
checkpoints.

In [ ]:
!rm -rf /content/smoke && mkdir -p /content/smoke
!python -m src.train.trainer \
    --config hindi/configs/train.yaml \
    --max-steps 100 \
    --ckpt-dir /content/smoke/ckpt \
    --log-jsonl /content/smoke/log.jsonl \
    --no-mirror

In [ ]:
import json

rows = [json.loads(l) for l in open('/content/smoke/log.jsonl')]
tr = [r for r in rows if 'train_loss' in r]
first, last = tr[0]['train_loss'], tr[-1]['train_loss']
tps = sum(r['tokens_per_sec'] for r in tr) / len(tr)

peak = max(r.get('peak_mem_gib', 0) for r in tr)
print(f'loss {first:.3f} -> {last:.3f}   ({tps:,.0f} tok/s)   peak GPU {peak:.1f} GiB')
print(f'ln(8000) = 8.987  (an untrained model sits here)')
assert last < first, 'loss did not decrease -- stop and investigate'
assert last < 8.987, 'loss above uniform-random -- something is wrong'

steps = 14329
print(f'\nprojected full run: {steps * 32768 / tps / 3600:.2f} hours')
print('SMOKE TEST PASSED')

## 5. Full run

~1.5-2.5 hours. Checkpoints land on Drive every 500 steps; validation every 500.

Leave the tab open — Colab kills idle sessions. If it dies anyway, re-run this
cell and it picks up from the last checkpoint.

In [ ]:
!python -m src.train.trainer --config hindi/configs/train.yaml

## 6. Verify what landed on Drive

In [ ]:
import json, glob, os

ck = f'{DRIVE_BASE}/hindi/checkpoints'
for f in sorted(glob.glob(f'{ck}/*.pt')):
    print(f'{os.path.basename(f):28} {os.path.getsize(f)/1e6:7.1f} MB')

rows = [json.loads(l) for l in open(f'{DRIVE_BASE}/hindi/logs/train.jsonl')]
tr = [r for r in rows if 'train_loss' in r]
va = [r for r in rows if 'val_loss' in r]
print(f'\n{len(tr)} train points, {len(va)} val points')
if va:
    best = min(va, key=lambda r: r['val_loss'])
    print(f"final train loss {tr[-1]['train_loss']:.4f}")
    print(f"best val loss    {best['val_loss']:.4f}  (ppl {best['val_ppl']:.2f}) at step {best['step']:,}")